# 02 - Preprocesamiento, Imputación e Ingeniería de Características
### SolFX-Predictor: Predicción del Tipo de Cambio USD/PEN
**Basado en la metodología de:**  
*Palma Arzubialde, C. y Salazar Sedano, J. G. (2025). Predicción multivariable del tipo de cambio USD/PEN mediante modelos de machine learning, deep learning y modelos fundacionales de series de tiempo.*

---
### 📌 Objetivos del Flujo:
1. **Limpieza e Inconsistencias:** Depurar errores de celda propios de exportaciones de Bloomberg (`#NAME?`).
2. **Alineación de Calendario e Imputación:** Sincronizar la serie al calendario hábil bursátil (`freq='B'`) e imputar días festivos o sin cotización mediante **Forward Fill (LOCF - Last Observation Carried Forward)** para evitar fuga de información (*lookahead bias*).
3. **Frecuencias Mixtas:** Fusionar series macroeconómicas mensuales (ej. subíndices de IPC) propagándolas de manera continua hacia adelante.
4. **Variable Objetivo:** Definición y cálculo del log-retorno diario continuo $r_t = \ln(P_t / P_{t-1})$.
5. **Pruebas de Estacionariedad:** Ejecución de contrastes Augmented Dickey-Fuller (ADF) y KPSS.
6. **Ingeniería de Características Anti-Leakage (Algoritmo 2 de la Tesis):**
   - Rezagos autorregresivos (AR): $r_{t-1}, r_{t-2}, r_{t-3}, r_{t-5}, r_{t-10}, r_{t-21}$.
   - Indicadores técnicos causales sobre precios en $t-1$: RSI(14) normalizado a $[-1, 1]$, MACD normalizado, Volatilidad de 21 días (proxy GARCH), distancia porcentual a medias móviles (SMA 5, 10, 20).
   - Variables temporales periódicas cíclicas: seno y coseno del mes y día de la semana.
7. **Partición Temporal Estricta:** División entre conjunto de entrenamiento y conjunto de *Holdout* de los últimos 60 días ($N_{holdout} = 60$) para evaluación fuera de muestra.
8. **Exportación:** Guardado de los datasets procesados en `data/processed/`.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

# Añadir src al path
sys.path.append(os.path.abspath('..'))
from src.preprocessing import SolFXPipeline, PreprocessingConfig

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Bibliotecas importadas correctamente.')

## 1. Carga de Datos Crudos de Bloomberg

In [ ]:
excel_path = '../data/raw/Datos_Bloomberg_PEN_FX.xlsx'
raw_pen_df = pd.read_excel(excel_path, sheet_name='Base_PEN')
monthly_macro_df = pd.read_excel(excel_path, sheet_name='IPC_mensual')

print(f'Base spot USD/PEN cruda: {raw_pen_df.shape}')
print(f'Base IPC mensual cruda:  {monthly_macro_df.shape}')
display(raw_pen_df.head())

## 2. Ejecución del Pipeline Automatizado

In [ ]:
config = PreprocessingConfig(
    date_col='Fecha',
    price_col='USD_PEN origen (PEN por USD)',
    target_col='PEN_log_ret',
    n_holdout=60,
    min_train_size=252,
    step_size=21
)

pipeline = SolFXPipeline(config)
full_df, train_df, holdout_df = pipeline.run_pipeline(
    raw_pen_df=raw_pen_df,
    monthly_macro_df=monthly_macro_df
)

saved_paths = pipeline.save_results(full_df, train_df, holdout_df, output_dir='../data/processed')
print('\nDatasets exportados:')
for k, v in saved_paths.items():
    print(f'  {k}: {v}')

## 3. Diagnóstico de Estacionariedad y Distribución del Target

In [ ]:
stats = pipeline.metadata.get('stationarity_tests', {})
print('Resultados de Contrastes de Estacionariedad:')
print(f"- ADF Estadístico: {stats.get('adf_stat'):.4f} (p-value: {stats.get('adf_pvalue'):.4e})")
print(f"  ¿Estacionario por ADF? {stats.get('adf_is_stationary')}")
print(f"- KPSS Estadístico: {stats.get('kpss_stat'):.4f} (p-value: {stats.get('kpss_pvalue'):.4f})")
print(f"  ¿Estacionario por KPSS? {stats.get('kpss_is_stationary')}")

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].plot(full_df['Fecha'], full_df['PEN_log_ret'], color='#1f77b4', lw=0.7)
axes[0].set_title('Evolución de Log-Retornos Diarios USD/PEN (1994 - 2026)')
axes[0].set_ylabel('Log-Retorno')

axes[1].hist(full_df['PEN_log_ret'], bins=100, color='#2ca02c', alpha=0.75, density=True)
axes[1].set_title('Distribución Empírica de Retornos')
axes[1].set_xlabel('Log-Retorno')
plt.tight_layout()
plt.show()

## 4. Visualización de Indicadores Técnicos Anti-Leakage

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

# Precios y SMA
axes[0].plot(full_df['Fecha'].iloc[-500:], full_df['USD_PEN'].iloc[-500:], label='USD/PEN Spot', color='black', lw=1.2)
axes[0].plot(full_df['Fecha'].iloc[-500:], full_df['SMA_20'].iloc[-500:], label='SMA 20 (Causal)', color='orange', ls='--')
axes[0].set_title('Precio Spot USD/PEN y Media Móvil Causal (Últimas 500 sesiones)')
axes[0].legend(loc='upper left')

# RSI Normalizado
axes[1].plot(full_df['Fecha'].iloc[-500:], full_df['RSI'].iloc[-500:], label='RSI Normalizado [-1, 1]', color='purple')
axes[1].axhline(0.4, color='red', linestyle=':', label='Sobrecompra (+0.4 / 70)')
axes[1].axhline(-0.4, color='green', linestyle=':', label='Sobreventa (-0.4 / 30)')
axes[1].set_title('RSI Normalizado Causal')
axes[1].legend(loc='lower left')

# Volatilidad Histórica
axes[2].plot(full_df['Fecha'].iloc[-500:], full_df['GARCH_vol'].iloc[-500:], label='Volatilidad Móvil 21d', color='darkred')
axes[2].set_title('Régimen de Volatilidad Histórica (Rolling 21d)')
axes[2].legend(loc='upper left')

plt.tight_layout()
plt.show()

## 5. Partición Temporal Estricta: Train vs Holdout

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(train_df['Fecha'].iloc[-250:], train_df['USD_PEN'].iloc[-250:], label='Entrenamiento (Último Año Bursátil)', color='#1f77b4')
plt.plot(holdout_df['Fecha'], holdout_df['USD_PEN'], label='Holdout Fuera de Muestra (Últimos 60 días)', color='#d62728', lw=2.2)
plt.axvline(train_df['Fecha'].iloc[-1], color='black', linestyle='--', label='Corte Train/Holdout')
plt.title('División Temporal Estricta: Train y Holdout Final')
plt.ylabel('Cotización USD/PEN')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Conjunto de Entrenamiento: {len(train_df)} filas ({train_df['Fecha'].min().date()} al {train_df['Fecha'].max().date()})")
print(f"Conjunto de Holdout:       {len(holdout_df)} filas ({holdout_df['Fecha'].min().date()} al {holdout_df['Fecha'].max().date()})")